<a href="https://colab.research.google.com/github/polreig/StartUp_DecoAI/blob/main/DecoAI_Pincel_Magico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalación de las herramientas

In [ ]:
!pip install -q -U google-genai diffusers transformers accelerate opencv-python gradio

El Cerebro y el Motor Gráfico

In [ ]:
import torch
from google import genai
from google.genai import types
from google.colab import userdata
from diffusers import StableDiffusionInpaintPipeline

print("1. Cargando credenciales de Gemini...")
try:
    GOOGLE_API_KEY = userdata.get('clave_API_gemini')
    gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
    print("✅ Gemini cargado correctamente.")
except Exception as e:
    print("⚠️ Error: Asegúrate de tener tu 'clave_API_gemini' en la llave del menú izquierdo de Colab.")

print("2. Cargando Motor de Inpainting Quirúrgico a la GPU...")
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
).to("cuda")

pipe.enable_attention_slicing()

print("✅ ¡Sistemas listos y estables!")

La Aplicación Web Modo Pincél

In [ ]:
import gradio as gr
import urllib.parse
from PIL import Image
import numpy as np
import json
import torch
from google.genai import types

# =========================================================
# FUNCIONES DE SOPORTE
# =========================================================
def preparar_imagen_pro(img, max_size=512):
    ancho, alto = img.size
    ratio = alto / ancho
    nuevo_ancho, nuevo_alto = (max_size, int(max_size * ratio)) if ancho > alto else (int(max_size / ratio), max_size)
    nuevo_ancho, nuevo_alto = (nuevo_ancho // 8) * 8, (nuevo_alto // 8) * 8
    return img.resize((nuevo_ancho, nuevo_alto), Image.Resampling.LANCZOS)

def generar_html_catalogo_mueble(datos_json):
    lista = sorted(datos_json.get('lista_compra_mueble', []), key=lambda x: x.get('precio_estimado', 0))
    html = "<div style='background: #fafafa; padding: 25px; border-radius: 10px; border: 1px solid #e9ecef;'><h2 style='color: #1a1a1a; margin: 0 0 15px 0;'>🛍️ Opciones de Compra</h2><div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); gap: 10px;'>"
    for item in lista:
        nombre = item.get('nombre', 'Mueble').title()
        query = urllib.parse.quote(nombre)
        precio = f"~{item.get('precio_estimado', 0)}€"
        html += f"<div style='background: white; border: 1px solid #eee; border-radius: 8px; padding: 14px;'><b>{nombre}</b> <span style='color: #2e7d32;'>({precio})</span><br><div style='margin-top: 8px; display: flex; gap: 5px;'><a href='https://www.ikea.com/es/es/search/?q={query}' target='_blank' style='background: #0058a3; color: white; padding: 5px 10px; border-radius: 3px; font-size: 10px; text-decoration: none;'>IKEA</a><a href='https://www.amazon.es/s?k={query}' target='_blank' style='background: #FF9900; color: #111; padding: 5px 10px; border-radius: 3px; font-size: 10px; text-decoration: none;'>Amazon</a><a href='https://www.leroymerlin.es/buscar?q={query}' target='_blank' style='background: #73c322; color: white; padding: 5px 10px; border-radius: 3px; font-size: 10px; text-decoration: none;'>Leroy Merlin</a><a href='https://www.zarahome.com/es/search.html?keyword={query}' target='_blank' style='background: #111; color: white; padding: 5px 10px; border-radius: 3px; font-size: 10px; text-decoration: none;'>Zara Home</a></div></div>"
    html += "</div></div>"
    return html

# =========================================================
# EL MOTOR QUIRÚRGICO (PARCHE GRADIO APLICADO)
# =========================================================
def motor_cambio_mueble_pincel(dict_imagen, peticion_mueble_es):
    print("\n--- NUEVA PETICIÓN INICIADA ---")
    
    if dict_imagen is None or not isinstance(dict_imagen, dict) or dict_imagen.get("background") is None:
        return "⚠️ Sube una imagen primero.", None, ""
        
    img_pil = dict_imagen["background"].convert("RGB")
    layers = dict_imagen.get("layers", [])
    
    if len(layers) > 0 and layers[0] is not None:
        mask_pil = layers[0].split()[-1].convert("L")
    else:
        return "⚠️ Pinta por encima del mueble viejo con el ratón.", [np.array(img_pil)], ""

    mask_array = np.array(mask_pil)
    if np.max(mask_array) == 0:
        return "⚠️ No has pintado nada. Pinta el mueble que quieres borrar.", [np.array(img_pil), np.array(mask_pil.convert("RGB"))], ""
    
    mask_array = np.where(mask_array > 0, 255, 0).astype(np.uint8)
    mask_pil_solida = Image.fromarray(mask_array)
    mask_pil_rgb = mask_pil_solida.convert("RGB") 

    print("✅ 1/4: Imágenes preparadas. Llamando a Gemini...")
    img_orig = preparar_imagen_pro(img_pil)
    mask_orig = preparar_imagen_pro(mask_pil_rgb) 
    
    prompt_brain = f"""
    Eres un diseñador de interiores. El cliente tapó un mueble y quiere: '{peticion_mueble_es}'. 
    DEVUELVE ÚNICAMENTE UN JSON VÁLIDO CON ESTA ESTRUCTURA EXACTA:
    {{
      "prompt_en_quirurgico": "A photorealistic room with a [descripción en inglés del mueble nuevo], perfectly blended, interior design, 8k",
      "lista_compra_mueble": [
        {{"nombre": "Nombre del mueble sugerido 1", "precio_estimado": 150}},
        {{"nombre": "Nombre del mueble sugerido 2", "precio_estimado": 200}},
        {{"nombre": "Nombre del mueble sugerido 3", "precio_estimado": 250}}
      ]
    }}
    """
    
    try:
        response = gemini_client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[img_orig, prompt_brain],
            config=types.GenerateContentConfig(response_mime_type="application/json")
        )
        datos = json.loads(response.text)
        catalogo_html = generar_html_catalogo_mueble(datos)
        print("✅ 2/4: Gemini ha respondido correctamente.")
    except Exception as e:
        print(f"❌ Error en Gemini: {e}")
        return f"⚠️ Error con Gemini: {e}", [np.array(img_orig), np.array(mask_orig)], ""
    
    print("✅ 3/4: Renderizando 3 opciones fotorrealistas con Inpainting...")
    torch.cuda.empty_cache()
    
    try:
        imagenes_generadas = pipe(
            prompt=datos.get('prompt_en_quirurgico', 'modern furniture, highly detailed') + ", photorealistic, high quality, interior design, perfectly integrated",
            negative_prompt="low quality, messy, floating, bad perspective, deformed, bad anatomy, cartoon",
            image=img_orig,
            mask_image=mask_orig, 
            num_inference_steps=25,
            guidance_scale=7.5,
            num_images_per_prompt=3 
        ).images
        print("✅ 4/4: ¡Renderizado completado! Procesando píxeles para la web...")
    except Exception as e:
        print(f"❌ Error en SD: {e}")
        return f"⚠️ Error en la IA gráfica: {e}", [np.array(img_orig), np.array(mask_orig)], ""

    # LA MAGIA QUE EVITA QUE GRADIO SE CUELGUE: Convertir a NumPy Array (Píxeles puros)
    galeria_final = [np.array(img_orig), np.array(mask_orig)]
    for img in imagenes_generadas:
        galeria_final.append(np.array(img))
    
    return f"✨ ¡Listo! He generado 3 opciones fotorrealistas para: **{peticion_mueble_es}**.", galeria_final, catalogo_html

# =========================================================
# INTERFAZ WEB SIMPLIFICADA
# =========================================================
with gr.Blocks(theme=gr.themes.Monochrome()) as app:
    gr.HTML("<div style='text-align: center; padding: 20px;'><h1 style='color: #111; font-size: 3em; margin-bottom: 0;'>🛋️ DECO.AI (Pincel Mágico)</h1><p style='color: #666;'>Sustituye un solo mueble de tu habitación.</p></div>")
    with gr.Row():
        with gr.Column(scale=1):
            in_img_editor = gr.ImageEditor(label="1. Sube tu foto y pinta el mueble", type="pil", image_mode="RGB")
            in_prompt_es = gr.Textbox(label="2. ¿Qué quieres poner ahí?", placeholder="Ej: Una mesa de cristal")
            btn = gr.Button("🪄 Cambiar Mi Mueble", variant="primary", size="lg")
            
        with gr.Column(scale=2): # Eliminamos el 'visible=False' que también colgaba Gradio
            out_analisis = gr.Markdown("Esperando instrucciones...")
            out_galeria = gr.Gallery(label="Original | Máscara | 3 Nuevas Opciones", columns=5, height="auto")
            out_catalogo = gr.HTML()

    # Ahora solo actualizamos 3 cosas (Texto, Galería, Catálogo)
    btn.click(
        fn=motor_cambio_mueble_pincel, 
        inputs=[in_img_editor, in_prompt_es], 
        outputs=[out_analisis, out_galeria, out_catalogo]
    )

app.queue().launch(share=True, debug=True)